# JAX-ALFA Simulation Run Log

Scans every `run.log` file under `examples*/` and reports when each simulation
was run, how long it took, and on which platform. Sort by **Completed** ascending
to identify the oldest (potentially stale) runs.

In [ ]:
from IPython.display import display, Markdown
from datetime import datetime, timezone
display(Markdown(f"*Last run: {datetime.now(timezone.utc).strftime('%B %d, %Y at %H:%M UTC')}*"))

In [ ]:
import os, re
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
from IPython.display import display, HTML

def find_repo_root():
    path = Path.cwd().resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'examples').is_dir() and (candidate / 'docs').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate jaxalfa repository root')

def read_elapsed(log_path, tail_bytes=4096):
    """Read Total Elapsed Time from the end of a (potentially large) log file."""
    try:
        size = os.path.getsize(log_path)
        with open(log_path, 'rb') as f:
            f.seek(max(0, size - tail_bytes))
            tail = f.read().decode('utf-8', errors='ignore')
        m = re.search(r'Total Elapsed Time:\s+([\d.]+)\s+seconds', tail)
        return float(m.group(1)) if m else None
    except Exception:
        return None

repo = find_repo_root()
print(f'Repository root: {repo}')

In [ ]:
rows = []

for log_path in sorted(repo.glob('examples*/*/runs/*/run.log')):
    rel   = log_path.relative_to(repo).parts
    # rel = (examples_A6000ada, SBL_GABLS4, runs, 256x256x256_LASDD_SM_SP, run.log)
    platform = rel[0]   # e.g. examples_A6000ada
    case     = rel[1]   # e.g. SBL_GABLS4
    run      = rel[3]   # e.g. 256x256x256_LASDD_SM_SP

    mtime     = os.path.getmtime(log_path)
    completed = datetime.fromtimestamp(mtime, tz=timezone.utc)
    elapsed   = read_elapsed(log_path)

    if elapsed is not None:
        started   = completed - timedelta(seconds=elapsed)
        wall_time = str(timedelta(seconds=int(elapsed)))
    else:
        started   = None
        wall_time = '—'

    rows.append({
        'Platform'       : platform,
        'Case'           : case,
        'Run'            : run,
        'Started (UTC)'  : started.strftime('%Y-%m-%d %H:%M') if started else '—',
        'Completed (UTC)': completed.strftime('%Y-%m-%d %H:%M'),
        'Wall Time'      : wall_time,
        '_completed_dt'  : completed,   # for sorting
    })

df_all = pd.DataFrame(rows)
print(f'{len(df_all)} runs found across {df_all["Platform"].nunique()} platform(s)')

## By Completion Date (oldest first)

Oldest runs appear at the top — these are candidates for re-running with the latest model.

In [ ]:
display_cols = ['Platform', 'Case', 'Run', 'Started (UTC)', 'Completed (UTC)', 'Wall Time']

df_by_date = (
    df_all.sort_values('_completed_dt')
          .reset_index(drop=True)
          [display_cols]
)
df_by_date.index += 1
display(df_by_date.style.set_properties(**{'text-align': 'left'}))

## By Platform → Case → Run

Same data grouped alphabetically by platform, case, and run name.

In [ ]:
df_by_name = (
    df_all.sort_values(['Platform', 'Case', 'Run'])
          .reset_index(drop=True)
          [display_cols]
)
df_by_name.index += 1
display(df_by_name.style.set_properties(**{'text-align': 'left'}))

## Summary by Platform

In [ ]:
summary = (
    df_all.groupby('Platform')
          .agg(
              Runs        = ('Run', 'count'),
              Oldest_Run  = ('_completed_dt', lambda x: x.min().strftime('%Y-%m-%d')),
              Newest_Run  = ('_completed_dt', lambda x: x.max().strftime('%Y-%m-%d')),
          )
          .rename(columns={'Oldest_Run': 'Oldest Run', 'Newest_Run': 'Newest Run'})
)
display(summary)